# Anticipez les besoins en consommation de bâtiments — Ville de Seattle

**Objectif** : prédire la consommation totale d'énergie (`SiteEnergyUse(kBtu)`) des bâtiments **non destinés à l'habitation**
à partir de leurs **données structurelles** (taille, usage, date de construction, localisation…), pour éviter des relevés coûteux.

Données : relevés 2016 du *Building Energy Benchmarking* de Seattle ([source](https://data.seattle.gov/Built-Environment/Building-Energy-Benchmarking-Data-2015-Present/teqw-tu6e/about_data)).

# Analyse Exploratoire

### Import des modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

### Analyse Exploratoire

In [ ]:
building_consumption = pd.read_csv("../data/2016_Building_Energy_Benchmarking.csv")
building_consumption.shape

In [ ]:
# On regarde comment un batiment est défini dans ce jeu de données
building_consumption.head()

In [ ]:
# On regarde le nombre de valeurs manquantes par colonne ainsi que leur type
building_consumption.info()

#### Sens des colonnes

| Famille | Colonnes | Rôle pour le projet |
|---|---|---|
| Identifiants | `OSEBuildingID`, `PropertyName`, `Address`, `TaxParcelIdentificationNumber` | Identifient un bâtiment, aucun pouvoir prédictif |
| Constantes | `DataYear` (2016), `City` (Seattle), `State` (WA) | Une seule valeur → inutiles |
| Type / usage | `BuildingType`, `PrimaryPropertyType`, `ListOfAllPropertyUseTypes`, `Largest/Second/ThirdLargestPropertyUseType` (+ `GFA`) | **Structurel** : usage du bâtiment et surface par usage |
| Localisation | `ZipCode`, `CouncilDistrictCode`, `Neighborhood`, `Latitude`, `Longitude` | **Structurel** |
| Bâti | `YearBuilt`, `NumberofBuildings`, `NumberofFloors`, `PropertyGFATotal`, `PropertyGFAParking`, `PropertyGFABuilding(s)` | **Structurel** : âge, taille (GFA = *Gross Floor Area*, en pieds²) |
| Score | `ENERGYSTARScore`, `YearsENERGYSTARCertified` | Score 1-100 **calculé à partir des consommations** relevées |
| Intensités | `SiteEUI`, `SourceEUI` (+ `WN` = corrigé météo) | Consommation / surface → **dérivées de la cible** |
| Consommations | `SiteEnergyUse(kBtu)` (**cible**), `SiteEnergyUseWN`, `SteamUse`, `Electricity`, `NaturalGas` | Issues du **relevé** |
| Émissions | `TotalGHGEmissions`, `GHGEmissionsIntensity` | Calculées à partir des consommations |
| Qualité | `DefaultData`, `ComplianceStatus`, `Outlier`, `Comments` | Qualité du relevé |

⚠️ **Fuite de données** : toutes les colonnes issues du relevé (intensités, consommations par source, émissions, score ENERGY STAR)
ne seront **pas disponibles** pour un bâtiment non encore mesuré. Elles servent à l'analyse mais seront **exclues du modèle**.

#### Choix de la variable cible

Plusieurs colonnes pourraient servir de cible. Il faut en choisir **une seule** pour tout le projet.

| Candidate | Signification | Verdict |
|---|---|---|
| `SiteEnergyUse(kBtu)` | Énergie totale consommée sur site en 2016 | ✅ **Retenue** : c'est exactement la « consommation totale d'énergie » demandée |
| `SiteEnergyUseWN(kBtu)` | Même chose, corrigée des conditions météo (*Weather Normalized*) | ❌ Valeur recalculée par modèle, pas la consommation réelle mesurée |
| `SiteEUI(kBtu/sf)` / `SiteEUIWN` | Consommation divisée par la surface | ❌ Intensité, pas consommation totale (c'est la cible divisée par la surface) |
| `SourceEUI(kBtu/sf)` / `SourceEUIWN` | Énergie primaire (pertes de production et de transport incluses) par surface | ❌ Dépend des coefficients de conversion de chaque source, pas seulement du bâtiment |
| `TotalGHGEmissions` / `GHGEmissionsIntensity` | Émissions de CO2 | ❌ Autre objectif (émissions), calculé à partir des consommations par source |

Une fois la cible choisie, **toutes les autres candidates deviennent des fuites de données** : elles sont calculées à partir du même relevé
et ne seront pas connues pour un bâtiment non mesuré.

In [ ]:
cible = "SiteEnergyUse(kBtu)"
candidates = [cible, "SiteEnergyUseWN(kBtu)", "SiteEUI(kBtu/sf)", "SourceEUI(kBtu/sf)", "TotalGHGEmissions"]
building_consumption[candidates].describe().T.round(1)

#### Journal des filtrages

On garde une trace du nombre de lignes avant et après chaque filtrage ou nettoyage.

In [ ]:
journal = []

def filtrer(df, masque, etape):
    """Garde les lignes où `masque` est vrai et enregistre l'étape dans le journal."""
    avant = len(df)
    df = df[masque]
    journal.append({"Étape": etape, "Avant": avant, "Après": len(df), "Retirées": avant - len(df)})
    print(f"{etape} : {avant} → {len(df)} ({avant - len(df)} retirées)")
    return df

#### Filtrage des bâtiments non destinés à l'habitation

In [ ]:
building_consumption["BuildingType"].value_counts()

Les types `Multifamily LR / MR / HR` sont des immeubles d'habitation : on les retire.
On conserve `NonResidential`, `Nonresidential COS` (bâtiments de la ville), `Nonresidential WA`, `SPS-District K-12` (écoles) et `Campus`.

In [ ]:
df = building_consumption.copy()
df = filtrer(df, ~df["BuildingType"].str.startswith("Multifamily"), "BuildingType non résidentiel")
df["PrimaryPropertyType"].value_counts()

Quelques bâtiments classés non résidentiels ont pourtant `Low-Rise Multifamily` comme usage principal : ce sont des logements, on les retire aussi.
Les `Residence Hall` (résidences universitaires) et `Senior Care Community` sont conservés : ce sont des établissements gérés comme des bâtiments tertiaires.

In [ ]:
df = filtrer(df, df["PrimaryPropertyType"] != "Low-Rise Multifamily", "Usage principal non résidentiel")

#### Qualité des relevés

In [ ]:
print(df["ComplianceStatus"].value_counts(), "\n")
print(df["Outlier"].value_counts())

In [ ]:
df.groupby("ComplianceStatus")[cible].describe()

- `Missing Data` : consommation à 0 → relevé inexploitable.
- `Non-Compliant` / `Error - Correct Default Data` : relevés jugés non conformes ou basés sur des valeurs par défaut par la ville.
- `Outlier` : relevés signalés comme aberrants (`Low` / `High outlier`) par la ville.

On ne garde que les relevés **conformes**, non signalés comme aberrants, avec une consommation **strictement positive** :
apprendre sur des relevés faux dégraderait le modèle.

In [ ]:
df = filtrer(df, df["ComplianceStatus"] == "Compliant", "Relevé conforme")
df = filtrer(df, df["Outlier"].isna(), "Non signalé outlier par la ville")
df = filtrer(df, df[cible] > 0, "Consommation > 0")

- Les relevés signalés `Outlier` et les consommations nulles étaient tous déjà non conformes : ces deux filtres ne retirent rien de plus,
  ils sont gardés comme garde-fous.
- ⚠️ Ce filtre retire **la quasi-totalité des écoles publiques** (`SPS-District K-12`) : 90 sur 98 sont en
  `Error - Correct Default Data` (relevé complété avec des valeurs par défaut). Les écoles restent représentées par les établissements
  classés `NonResidential` et `Campus`, mais le modèle sera moins fiable sur ce type de bâtiment.

#### Suppression des colonnes inutiles

In [ ]:
colonnes_inutiles = [
    "DataYear", "City", "State",                     # constantes
    "PropertyName", "Address", "TaxParcelIdentificationNumber",  # identifiants texte
    "Comments",                                      # 100 % vide
    "YearsENERGYSTARCertified",                      # > 90 % vide
    "ComplianceStatus", "Outlier", "DefaultData",    # qualité, déjà utilisées pour filtrer
]
df = df.drop(columns=colonnes_inutiles).set_index("OSEBuildingID")
df.shape

#### Valeurs manquantes

In [ ]:
manquants = df.isna().sum()
(manquants[manquants > 0] / len(df) * 100).round(1).sort_values(ascending=False).to_frame("% manquant")

On ne supprime **pas** toutes les lignes ayant une valeur manquante : il ne resterait presque plus de bâtiments.
- `Second/ThirdLargestPropertyUseType` : manquants **par construction** (le bâtiment n'a qu'un ou deux usages) → on les remplacera par « aucun » / 0 m².
- `ENERGYSTARScore` : environ un tiers manquant, et **calculé à partir des consommations** → exclu du modèle.
- Les rares manquants restants (`ZipCode`, `LargestPropertyUseType`…) seront traités en préparation des features.

#### Cohérence des données

In [ ]:
# Quartiers : la casse n'est pas homogène (« North » / « NORTH »)
df["Neighborhood"] = df["Neighborhood"].str.upper().replace("DELRIDGE NEIGHBORHOODS", "DELRIDGE")
df["Neighborhood"].value_counts()

In [ ]:
ratio_usage = df["LargestPropertyUseTypeGFA"] / df["PropertyGFATotal"]
print("NumberofBuildings = 0 :", (df["NumberofBuildings"] == 0).sum())
print("NumberofFloors = 0    :", (df["NumberofFloors"] == 0).sum())
print("Surface usage principal > surface totale :", (ratio_usage > 1).sum(),
      "— dont écart > 10 % :", (ratio_usage > 1.1).sum())
df.sort_values("NumberofFloors", ascending=False)[["PrimaryPropertyType", "Neighborhood", "NumberofFloors", "PropertyGFATotal"]].head(5)

- `NumberofBuildings = 0` est impossible pour une propriété relevée : c'est un oubli de saisie, la propriété compte au moins
  un bâtiment → **corrigé à 1** (supprimer ces lignes ferait perdre des bâtiments par ailleurs complets).
- `NumberofFloors = 0` : impossible, et on ne peut pas deviner le vrai nombre d'étages (on y trouve aussi bien un hôtel de 90 000 m²
  qu'un campus de 111 bâtiments) → **supprimés**.
- Un lieu de culte de **99 étages** pour 2 000 m² : le plus haut gratte-ciel de Seattle (Columbia Center) en compte 76 → **supprimé**.
- La surface de l'usage principal dépasse la surface totale pour environ 180 bâtiments. Un petit écart peut venir d'arrondis
  ou de déclarations séparées ; au-delà de **10 %**, les surfaces sont incohérentes → **supprimés**.

In [ ]:
df.loc[df["NumberofBuildings"] == 0, "NumberofBuildings"] = 1
df = filtrer(df, df["NumberofFloors"] > 0, "NumberofFloors > 0")
df = filtrer(df, df["NumberofFloors"] < 99, "Nombre d'étages plausible")
ratio_usage = df["LargestPropertyUseTypeGFA"] / df["PropertyGFATotal"]
df = filtrer(df, ~(ratio_usage > 1.1), "Surface usage principal ≤ 110 % surface totale")

`~(ratio > 1.1)` plutôt que `ratio <= 1.1` : les bâtiments sans usage principal renseigné (ratio manquant) sont ainsi **conservés**.

In [ ]:
pd.DataFrame(journal)

#### Statistiques descriptives

In [ ]:
colonnes_num = ["YearBuilt", "NumberofBuildings", "NumberofFloors", "PropertyGFATotal",
                "PropertyGFAParking", "PropertyGFABuilding(s)", "ENERGYSTARScore",
                "SiteEUI(kBtu/sf)", cible, "TotalGHGEmissions"]
df[colonnes_num].describe().T.round(1)

#### Distribution de la cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df[cible], bins=60, ax=axes[0])
axes[0].set_title("SiteEnergyUse (kBtu)")
sns.histplot(np.log10(df[cible]), bins=60, kde=True, ax=axes[1])
axes[1].set_title("log10(SiteEnergyUse)")
plt.tight_layout()
plt.show()
print(f"Asymétrie brute : {df[cible].skew():.1f} — après log : {np.log10(df[cible]).skew():.2f}")

La consommation est **extrêmement asymétrique** : la plupart des bâtiments consomment quelques millions de kBtu,
quelques très gros sites (campus, hôpitaux) des centaines de millions. En **échelle log**, la distribution devient proche d'une gaussienne :
on modélisera probablement `log(consommation)`, sinon les quelques géants domineraient l'erreur du modèle.

In [ ]:
df.sort_values(cible, ascending=False)[["PrimaryPropertyType", "NumberofBuildings", "PropertyGFATotal", cible]].head(10)

#### Consommation selon l'usage

In [ ]:
ordre = df.groupby("PrimaryPropertyType")[cible].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 7))
sns.boxplot(data=df, y="PrimaryPropertyType", x=cible, order=ordre, log_scale=True, ax=ax)
ax.set_title("Consommation totale par type de bâtiment (échelle log)")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ordre_eui = df.groupby("PrimaryPropertyType")["SiteEUI(kBtu/sf)"].median().sort_values(ascending=False).index
sns.boxplot(data=df, y="PrimaryPropertyType", x="SiteEUI(kBtu/sf)", order=ordre_eui, log_scale=True, ax=ax)
ax.set_title("Intensité énergétique (kBtu / pied²) par type de bâtiment")
plt.show()

- En valeur absolue, les **hôpitaux, laboratoires et grands bureaux** consomment le plus : ce sont aussi les plus grands.
- Ramenée à la surface (*EUI*), la hiérarchie change : **supermarchés, restaurants, hôpitaux et laboratoires** sont les plus intensifs
  (froid, cuisson, équipements 24 h/24), **entrepôts et self-stockage** les plus sobres.
- → l'**usage** du bâtiment est une information clé, à combiner avec la **surface**.

#### Consommation et surface

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df, x="PropertyGFATotal", y=cible, alpha=0.5, ax=ax)
ax.set(xscale="log", yscale="log", title="Consommation vs surface totale (log-log)")
plt.show()
print("Corrélation log-log :", round(np.corrcoef(np.log10(df["PropertyGFATotal"]), np.log10(df[cible]))[0, 1], 2))

Relation **quasi linéaire en log-log** : la surface est la variable la plus explicative. La dispersion verticale autour de la tendance
correspond aux différences d'usage vues plus haut.

#### Âge, hauteur et localisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["YearBuilt"], bins=40, ax=axes[0]).set_title("Année de construction")
sns.histplot(df["NumberofFloors"], bins=40, ax=axes[1]).set_title("Nombre d'étages")
axes[1].set_yscale("log")
sns.scatterplot(data=df, x="YearBuilt", y="SiteEUI(kBtu/sf)", alpha=0.4, ax=axes[2]).set_title("Intensité vs année")
axes[2].set_yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc = axes[0].scatter(df["Longitude"], df["Latitude"], c=np.log10(df[cible]), cmap="viridis", s=12)
plt.colorbar(sc, ax=axes[0], label="log10(consommation)")
axes[0].set_title("Localisation des bâtiments")
ordre_q = df.groupby("Neighborhood")[cible].median().sort_values(ascending=False).index
sns.boxplot(data=df, y="Neighborhood", x=cible, order=ordre_q, log_scale=True, ax=axes[1])
axes[1].set_title("Consommation par quartier")
plt.tight_layout()
plt.show()

- Le parc est **ancien** (médiane en 1965, un quart des bâtiments construits avant 1930, pics dans les années 1920 et 1960) ;
  l'année de construction n'est que **faiblement** liée à l'intensité énergétique (Spearman ≈ 0,19).
- Les trois quarts des bâtiments font moins de 5 étages ; les tours (20 étages et plus) sont presque toutes à **Downtown**.
- **Downtown** et **Lake Union** regroupent les bâtiments les plus consommateurs (grands bureaux, hôtels, hôpitaux) :
  l'effet quartier reflète surtout le type et la taille des bâtiments.

#### Sources d'énergie

In [ ]:
sources = {"Électricité": "Electricity(kBtu)", "Gaz naturel": "NaturalGas(kBtu)", "Vapeur": "SteamUse(kBtu)"}
mix = pd.DataFrame({nom: df[col] / df[cible] for nom, col in sources.items()})
mix["PrimaryPropertyType"] = df["PrimaryPropertyType"]
mix_type = mix.groupby("PrimaryPropertyType").mean().sort_values("Électricité")
mix_type.plot(kind="barh", stacked=True, figsize=(10, 7), title="Part moyenne de chaque source d'énergie")
plt.xlabel("Part de la consommation totale")
plt.show()
print("Part moyenne sur l'ensemble :", mix[list(sources)].mean().round(2).to_dict())
print("Bâtiments utilisant du gaz :", (df["NaturalGas(kBtu)"] > 0).mean().round(2),
      "— de la vapeur :", (df["SteamUse(kBtu)"] > 0).mean().round(2))

L'**électricité** domine (environ 70 % de la consommation en moyenne). Le **gaz** dépasse la moitié de la consommation dans les
écoles, lieux de culte et restaurants, et en représente près de la moitié dans les résidences seniors (chauffage, cuisson) ; la **vapeur** (réseau urbain) ne concerne que 7 % des
bâtiments, essentiellement à Downtown et dans le quartier East (hôpitaux de First Hill). Ces informations proviennent du relevé : elles éclairent l'analyse mais ne seront pas utilisées comme features.

#### Type de bâtiment et quartier (quali vs quali)

In [ ]:
croise = pd.crosstab(df["PrimaryPropertyType"], df["Neighborhood"])
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(croise, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Nombre de bâtiments par type et par quartier")
plt.show()

Deux variables qualitatives se comparent avec un **tableau croisé** (ici en heatmap).
- Les **grands bureaux** et les **hôtels** sont concentrés à **Downtown** (environ 60 % et 55 %) et à **Lake Union**.
- Les **entrepôts** et **centres de distribution** sont massivement à **Greater Duwamish** (zone industrielle et portuaire) :
  environ 70 % des entrepôts et 80 % des centres de distribution.
- Les **lieux de culte** et **supermarchés** sont répartis dans les quartiers résidentiels (Northeast, Northwest, Magnolia / Queen Anne).

Le quartier et le type de bâtiment ne sont donc pas indépendants : l'« effet quartier » sur la consommation est en partie un effet type.

#### Corrélations

In [ ]:
cols_corr = ["YearBuilt", "NumberofBuildings", "NumberofFloors", "PropertyGFATotal", "PropertyGFAParking",
             "PropertyGFABuilding(s)", "LargestPropertyUseTypeGFA", "ENERGYSTARScore",
             "SiteEUI(kBtu/sf)", "SiteEnergyUseWN(kBtu)", "Electricity(kBtu)", "NaturalGas(kBtu)",
             "TotalGHGEmissions", cible]
corr = df[cols_corr].corr(method="spearman")
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Corrélations de Spearman")
plt.show()

On utilise Spearman (corrélation des rangs), plus adaptée à des variables aussi asymétriques.
- `SiteEnergyUseWN`, `Electricity`, `TotalGHGEmissions` sont très fortement corrélées à la cible (0,86 à 0,98) : ce sont des **mesures de la même chose** → fuite de données, à exclure.
- Les surfaces `PropertyGFATotal`, `PropertyGFABuilding(s)`, `LargestPropertyUseTypeGFA` sont **redondantes** entre elles (> 0,9) : il faudra n'en garder qu'une partie.
- Parmi les variables structurelles, la **surface** est de loin la plus corrélée à la consommation ; (≈ 0,74) ; viennent ensuite le nombre d'étages (≈ 0,48), la surface de parking (≈ 0,38) et l'année de construction (≈ 0,29).

#### Synthèse de l'analyse exploratoire

1. **Périmètre** : **1 454 bâtiments** non résidentiels (sur 3 376 au départ) avec un relevé conforme, une consommation positive et des
   caractéristiques cohérentes (voir le journal des filtrages). Les écoles publiques sont sous-représentées.
2. **Cible très asymétrique** → modélisation sur `log(consommation)` à envisager, et vigilance sur les très gros sites (campus, hôpitaux).
3. **Surface** = premier facteur de consommation (relation linéaire en log-log).
4. **Usage** = second facteur : à surface égale, un supermarché ou un hôpital consomme bien plus qu'un entrepôt.
5. **Localisation et âge** ont un effet faible ou indirect (Downtown concentre les grands bâtiments).
6. **Fuite de données** : intensités, consommations par source, émissions et ENERGY STAR Score sont issues du relevé → exclues des features.

# Modélisation 

### Import des modules 

In [ ]:
#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

### Feature Engineering

A réaliser : Enrichir le jeu de données actuel avec de nouvelles features issues de celles existantes. 

In [ ]:
# CODE FEATURE ENGINEERING

### Préparation des features pour la modélisation

A réaliser :
* Si ce n'est pas déjà fait, supprimer toutes les colonnes peu pertinentes pour la modélisation.
* Tracer la distribution de la cible pour vous familiariser avec l'ordre de grandeur. En cas d'outliers, mettez en place une démarche pour les supprimer.
* Débarrassez-vous des features redondantes en utilisant une matrice de corrélation.
* Réalisez différents graphiques pour comprendre le lien entre vos features et la target (boxplots, scatterplots, pairplot si votre nombre de features numériques n'est pas très élevé).
*  Séparez votre jeu de données en un Pandas DataFrame X (ensemble de feautures) et Pandas Series y (votre target).
* Si vous avez des features catégorielles, il faut les encoder pour que votre modèle fonctionne.

In [ ]:
# CODE PREPARATION DES FEATURES

### Comparaison de différents modèles supervisés

A réaliser :
* Pour chaque algorithme que vous allez tester, vous devez :
    * Réaliser au préalable une séparation en jeu d'apprentissage et jeu de test via une validation croisée.
    * Si les features quantitatives que vous souhaitez utiliser ont des ordres de grandeur très différents les uns des autres, et que vous utilisez un algorithme de regression qui est sensible à cette différence, alors il faut réaliser un scaling (normalisation) de la donnée au préalable.
    * Entrainer le modèle sur le jeu de Train
    * Prédire la cible sur la donnée de test (nous appelons cette étape, l'inférence).
    * Calculer les métriques de performance R2, MAE et RMSE sur le jeu de train et de test.
    * Interpréter les résultats pour juger de la fiabilité de l'algorithme.
* Vous pouvez choisir par exemple de tester un modèle linéaire, un modèle à base d'arbres et un modèle de type SVM
* Déterminer le modèle le plus performant parmi ceux testés.

In [ ]:
# CODE COMPARAISON DES MODELES

### Optimisation et interprétation du modèle

A réaliser :
* Reprennez le meilleur algorithme que vous avez sécurisé via l'étape précédente, et réalisez une GridSearch de petite taille sur au moins 3 hyperparamètres.
* Si le meilleur modèle fait partie de la famille des modèles à arbres (RandomForest, GradientBoosting) alors utilisez la fonctionnalité feature importance pour identifier les features les plus impactantes sur la performance du modèle. Sinon, utilisez la méthode Permutation Importance de sklearn.

In [ ]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE